# import libraries

In [ ]:
import os
import pandas as pd 
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


# read data

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")

In [ ]:
print('shape of train data', train.shape)
print('columns of train data', train.columns)
print('type of train data', train.dtypes)

In [ ]:
print('shape of test data', test.shape)
print('columns of test data', test.columns)
print('type of test data', test.dtypes)


In [ ]:
train.head(3)

In [ ]:
test.head(3)

In [ ]:
train.drop(columns=["id"], inplace=True)

In [ ]:
train.info(memory_usage='deep')

In [ ]:
test.info(memory_usage='deep')

In [ ]:
print('train data')
print(train.isna().sum()) #checking if train has null
print(50*'-')
print('test data')
print(test.isna().sum()) #checking if test has null

In [ ]:
cat = ['spectral_type', 'galaxy_population', 'class']
print('train data')
for i in cat:
    print(f'{i}: {train[i].unique()}')
    
print(50*'-')

print('test data')
for i in cat[:-1]:
    print(f'{i}: {test[i].unique()}')

In [ ]:
print('train data')
for i in cat:
    print(train[i].value_counts(),'\n')


print(50*'-')
print('test data')
for i in cat[:-1]:
    print(test[i].value_counts(),'\n')

# Model

## split data

In [ ]:
X = train.iloc[:,:-1]
y = train.iloc[:,-1]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)


In [ ]:
print(f"Train: {(X_train.shape)}, Validation: {(X_val.shape)}")

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

In [ ]:
cat_features = ['spectral_type', 'galaxy_population']

model = CatBoostClassifier(
    iterations=2000,
    depth=10,
    learning_rate=0.03,
    l2_leaf_reg=3,
    border_count=254,
    random_strength=2,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    early_stopping_rounds=300,
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=200)

# Train
model.fit(
    X_train,
    y_train,
    cat_features = cat_features,
    eval_set = (X_val, y_val),
    use_best_model = True)

# Validation Prediction
pred_val = model.predict(X_val)

# Evaluation
print("\nBalanced Accuracy:")
print(balanced_accuracy_score(y_val, pred_val))

print("\nClassification Report:")
print(classification_report(y_val, pred_val))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, pred_val))

In [ ]:
y_val_decoded = le.inverse_transform(y_val)
pred_val_decoded = le.inverse_transform(pred_val)

ConfusionMatrixDisplay.from_predictions(
    y_val_decoded,
    pred_val_decoded,
    labels=['GALAXY', 'QSO', 'STAR'],  
    cmap="Blues",
    values_format="d"
)

plt.title("Confusion Matrix")
plt.show()

## prediction

In [ ]:
test_ids = test.id
test.drop(columns=["id"], inplace=True)

In [ ]:
X = train.iloc[:,:-1]
y = train.iloc[:,-1]

le = LabelEncoder()
y = le.fit_transform(y)

cat_features = ['spectral_type', 'galaxy_population']


model.fit(X, y, cat_features=cat_features)

pred_test = model.predict(test)

In [ ]:
pred_test = pred_test.astype(int).ravel()

submission = pd.DataFrame({
    "id": test_ids,
    "class": le.inverse_transform(pred_test)
})

submission.to_csv("submission.csv", index=False)

In [ ]:
submission.head()